# Predicting Survival on the Titanic: A Logistic Regression Analysis

## 1. Introduction

### Problem Statement
The sinking of the Titanic is one of the most infamous shipwrecks in history. While there was an element of luck involved in surviving, it seems some groups of people were more likely to survive than others. The goal of this project is to build a highly interpretable logistic regression model to predict whether a passenger survived the disaster.

### Project Goals
1.  **Explore the Data:** Perform a thorough Exploratory Data Analysis (EDA) to understand the relationships between various passenger attributes and their survival outcome.
2.  **Engineer Features:** Create new, meaningful features from the existing data to improve model performance, reliabilty and gather valuable insight.
3.  **Build and Validate a Model:** Construct a stable and accurate Logistic Regression model, paying close attention to the model's underlying assumptions.
4.  **Interpret the Results:** Use the final model to draw clear, data-driven conclusions about the key factors that determined survival on the Titanic.

### Tech Stack
-   Pandas for data manipulation
-   NumPy for numerical operations
-   Matplotlib & Seaborn for data visualization
-   Scikit-learn for model building and evaluation
-   Statsmodels for statistical analysis (VIF)

## 2. Exploratory Data Analysis (EDA)

In this section, I will explore the dataset to understand the characteristics of the passengers and identify potential predictors of survival. I will start with univariate analysis to examine each feature individually, followed by bivariate analysis to see how they relate to the `Survived` variable.

### 2.1 Univariate Analysis
Let's first look at the distribution of each variable on its own. Will use statistical tools like Kernel density functions, QQ plot and tests like Shapiro-Wilk to confirm there distribution (for numerical columns).

#### Reading the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(r"C:\Users\Sakshit Attri\Desktop\Titanic-Dataset.csv")

#### Survived columns

In [ ]:
#Survived columns
freq_Survived = df["Survived"].value_counts()
freq_Survived.plot(kind="bar")
plt.xlabel("Category")
plt.ylabel("Frequency")
plt.title("Category Frequency Distribution")
plt.show()

#Probability
probability_of_surviving = len(df[df["Survived"] == 1])/len(df)

Inference: The probability of getting survived in general is less.

#### PClass columns

In [ ]:
#PClass columns
freq_Pclass = df["Pclass"].value_counts()
freq_Pclass.plot(kind="bar")
plt.xlabel("Category")
plt.ylabel("Frequency")
plt.title("Category Frequency Distribution")
plt.show()

Inference: The probability of selecting a random person : class 3 > class 1 > class 2

#### Sex column

In [ ]:
#Sex column
freq_Sex = df["Sex"].value_counts()
freq_Sex.plot(kind="bar")
plt.xlabel("Category")
plt.ylabel("Frequency")
plt.title("Category Frequency Distribution")
plt.show()

Inference: The probability of selecting a random person : Male passengers > Female

#### Age column

In [ ]:
#Checking for nan values
df['Age'].isna().any()

In [ ]:
#Measure of centeral tendency
mean = df['Age'].mean()
median = df['Age'].median()

In [ ]:
#Measure of dispersion
variance = df['Age'].var()
standard_deviation = df['Age'].std()

In [ ]:
#5 number summary
five_num = {
    "Minimum": df['Age'].min(),
    "Q1 (25%)": df['Age'].quantile(0.25),
    "Median (50%)": df['Age'].median(),
    "Q3 (75%)": df['Age'].quantile(0.75),
    "Maximum": df['Age'].max()
}

In [ ]:
#Plotting the probability distribution curve for the data
import seaborn as sns
sns.histplot(df['Age'], bins=6, kde=True, stat="probability")  
plt.title("Empirical Probability Distribution")
plt.xlabel("Value")
plt.ylabel("Probability")
plt.show()

# Q-Q plot against Normal distribution
import scipy.stats as stats
stats.probplot(np.array(df['Age']), dist="norm", sparams=(1,), plot=plt)
plt.title("Q-Q Plot (Normal Distribution)")
plt.show()

#Checking whether the data is log normal or not using Shapiro Wilk test
epsilon = 1e-10
from scipy.stats import shapiro
log_data = np.log(df['Age'].dropna().reset_index(drop = True) + epsilon ) 
stat, p = shapiro(log_data)
print("Shapiro-Wilk test: stat=%.4f, p=%.4f" % (stat, p))

Inference
1. Most people lie in age group of 20.125 - 38.0 with median 28 and mean 29.
2. From probability distribution function, the data is not normally distributed rather seemed to be right skewed.
3. The "Non-Normality" of the data is also proved from Shapiro-wilk Test since p < 0.05.
4. **The data are missing in this column, need to hypertune the Imputation process during model building**

#### SibSp and Parch

In [ ]:
#SibSp
freq_SibSp = df["SibSp"].value_counts()
freq_SibSp.plot(kind="bar")
plt.xlabel("Category")
plt.ylabel("Frequency")
plt.title("Category Frequency Distribution")
plt.show()

#### Parch

In [ ]:
#Parch
freq_Parch = df["Parch"].value_counts()
freq_Parch.plot(kind="bar")
plt.xlabel("Category")
plt.ylabel("Frequency")
plt.title("Category Frequency Distribution")
plt.show()

Inference : 
1. Passengers travelling with Siblings and Spouces 0 > 1 > 2 > 4 > 3 > 8 > 5.
2. Passengers travelling with Siblings and Spouces 0 > 1 > 2 > 5 > 3 > 4 > 6.
3. **Can try making one column for all both SibSip and Parch. Before that need to analyse individual relationship with Survived column**

#### Fare

In [ ]:
#Checking for nan values
df['Fare'].isna().any()

In [ ]:
#Measure of centeral tendency
mean = df['Fare'].mean()
median = df['Fare'].median()

In [ ]:
#Measure of dispersion
variance = df['Fare'].var()
standard_deviation = df['Fare'].std()

In [ ]:
#5 number summary
five_num = {
    "Minimum": df['Fare'].min(),
    "Q1 (25%)": df['Fare'].quantile(0.25),
    "Median (50%)": df['Fare'].median(),
    "Q3 (75%)": df['Fare'].quantile(0.75),
    "Maximum": df['Fare'].max()
}

In [ ]:
#Plotting the probability distribution curve for the data
import seaborn as sns
sns.histplot(df['Fare'], bins=6, kde=True, stat="probability")  
plt.title("Empirical Probability Distribution")
plt.xlabel("Value")
plt.ylabel("Probability")
plt.show()

# Q-Q plot against Normal distribution
import scipy.stats as stats
stats.probplot(np.array(df['Fare']), dist="norm", sparams=(1,), plot=plt)
plt.title("Q-Q Plot (Normal Distribution)")
plt.show()

#Assuming Pareto distribution, converting to normal distribution using log
epsilon = 1e-10
x_transformed = np.log(df['Fare'] + epsilon)

#Checking whether the data is pareto or not using Shapiro Wilk test
from scipy.stats import shapiro
stat, p = shapiro(x_transformed)
print("Shapiro-Wilk test: stat=%.4f, p=%.4f" % (stat, p))

Inference : 
1. The data is Very Right skewed which actually tells 80:20% rule in wealth. 
2. It is not normally distributed and neither completely pareto distribution as seen using Shapiro test.

#### Embarked

In [ ]:
df["Embarked"].isna().any()

In [ ]:
#Embarked
freq_Embarked = df["Embarked"].value_counts()
freq_Embarked.plot(kind="bar")
plt.xlabel("Category")
plt.ylabel("Frequency")
plt.title("Category Frequency Distribution")
plt.show()

Inference : 
1. Frequency S > C > Q.
2. **Since we have NaN rows, need to do some feature engineering**

#### Feature engineering after Univariate Analysis
1. I found some columns were not usefull and did not have much information for modelling `Survived` column. We are removing these Un-wanted columns.
2. We are removing duplicates in the data if any.
3. We are creating 2 more features for `Age` column (mean imputation and median imputation) whose performance will be tested during model evaluation phase.
4. The `SibSp` and `Parch` columns both relate to a passenger's family. We can combine them into a single `FamilySize` feature and then categorize it. This will be more powerful than using the two features separately.
5. The `Embarked` column needs data imputation due to NaN values, therefore we created 2 more column named `Embarked_Mode_Imputing` and `Embarked_Missing_Imputing` whose performance will be tested during model evalutaion phase

In [ ]:
#Dropping the Un-wanted columns from the dataframe
df = df.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1).reset_index(drop = True)

#Dropping the duplicates
df = df.drop_duplicates().reset_index(drop = True)

#Age column Imputation
df['Age_Mean_Imputed'] = df['Age'].fillna(df['Age'].mean())
df['Age_Median_Imputed'] = df['Age'].fillna(df['Age'].median())

#SibSp and Parch combined
df['Total Family Size'] = df['SibSp'] + df['Parch'] + 1

#Embarked
df['Embarked_Mode_Imputing'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Embarked_Missing_Imputing'] = df['Embarked'].fillna('Missing')

Note : Will try doing K-Imputing once we finalize the dataset

### 2.2 Bivariate Analysis

Now that we understand the individual features, let's explore their relationship with the target variable, `Survived`. We will use statistical tests (like the Chi-Squared test for categorical features and ANOVA for numerical features) to confirm if the observed relationships are statistically significant.

#### Pclass column and Survival

In [ ]:
#Probability of surviving in different Pclasses
class_3_df = df[df['Pclass'] == 3].reset_index(drop = True)
probability_of_surviving_in_3class = len(class_3_df[class_3_df['Survived'] == 1])/len(class_3_df)
print(f"Probability of surviving if Pclass = 3 : ", probability_of_surviving_in_3class)
class_2_df = df[df['Pclass'] == 2].reset_index(drop = True)
probability_of_surviving_in_2class = len(class_2_df[class_2_df['Survived'] == 1])/len(class_2_df)
print(f"Probability of surviving if Pclass = 2 : ", probability_of_surviving_in_2class)
class_1_df = df[df['Pclass'] == 1].reset_index(drop = True)
probability_of_surviving_in_1class = len(class_1_df[class_1_df['Survived'] == 1])/len(class_1_df)
print(f"Probability of surviving if Pclass = 1 : ", probability_of_surviving_in_1class)

#Confirming with Chi-Squared-test (Test for Independence)
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df["Pclass"], df["Survived"])
_, p, _, expected = chi2_contingency(contingency_table)
print("p-value =", p)

#Performing Standard Residual to confirm that our observation is correct
std_residual = (contingency_table - expected)/np.sqrt(expected)
print(std_residual)

Inference on Pclass column
1. From probability : probability of surviving : Pclass 1 > Pclass 2 > Pclass 3.

2. From Chi-Squared test : p-value is extremely small 4.54e-23 way less than significance level of 0.01. Hence, we can reject null hypothesis and thus assume that "survival" is strongly associated with passenger "Pclass". 

3. From standard residual, it can be infered that:
    - Class 1 had higher probability of survival. (0 -> -4.6 whose abs() is > 2 and 1 -> 5.83 which is > 2). Both term imply same thing.
    - Class 2, can't infere much
    - Class 3 had less probability of survival. (0 -> 3.99 which is > 2 and 1 -> -5.059 whose abs() is > 2). Both term imply same thing.

#### Sex column and Survival

In [ ]:
#Probability of surviving in difference Pclasses
class_male_df = df[df['Sex'] == 'male'].reset_index(drop = True)
probability_of_surviving_in_male_class = len(class_male_df[class_male_df['Survived'] == 1])/len(class_male_df)
print(f"Probability of surviving if male : ", probability_of_surviving_in_male_class)
class_female_df = df[df['Sex'] == 'female'].reset_index(drop = True)
probability_of_surviving_in_female_class = len(class_female_df[class_female_df['Survived'] == 1])/len(class_female_df)
print(f"Probability of surviving if female : ", probability_of_surviving_in_female_class)

#Confirming with Chi-Squared-test (Test for Independence)
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df["Sex"], df["Survived"])
_, p, _, expected = chi2_contingency(contingency_table)
print("p-value =", p)

#Performing Standard Residual to confirm that our observation is correct
std_residual = (contingency_table - expected)/np.sqrt(expected)
print(std_residual)

Inference on Pclass column
1. From probability : probability of surviving : Female > Male

2. From Chi-Squared test : p-value is extremely small 1.197e-58 way less than significance level of 0.01. Hence, we can reject null hypothesis and thus assume that "Sex" is strongly associated with passenger "Pclass". 

3. From standard residual, it can be infered that:
    - Female had higher probability of survival. (0 -> -8.08 whose abs() is > 2 and 1 -> 10.24 which is > 2). Both term imply same thing.
    - Male had less probability of survival. (0 -> 5.96 which is > 2 and 1 -> -7.55 whose abs() is > 2). Both term imply same thing.

#### SibSp column and Survival

In [ ]:
all_unique_SibSp = df['SibSp'].unique()
for i in range(0, len(all_unique_SibSp)):
    SibSp_valv = df[df['SibSp'] == all_unique_SibSp[i]].reset_index(drop = True)
    probability_of_SibSp_valv_class = len(SibSp_valv[SibSp_valv['Survived'] == 1])/len(SibSp_valv)
    print(f"SibSp -> {all_unique_SibSp[i]}, Probability of surviving -> {probability_of_SibSp_valv_class*100}")
    
#Confirming with Chi-Squared-test (Test for Independence)
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df["SibSp"], df["Survived"])
_, p, _, expected = chi2_contingency(contingency_table)
print("p-value =", p)

#Performing Standard Residual to confirm that our observation is correct
std_residual = (contingency_table - expected)/np.sqrt(expected)
print(std_residual)

Inference
1. Using chi-squared test for independence of variables, we got p-value = 1.55e-06 which tell high dependence of SibSp with the Survived column.
2. Using the standard residual column, there is significant relationship when SibSp = 1, other than that, we are not able to see very high levels of significance with other values of SibSp.
    - In future while doinf feature engineering, we might change the feature in some way so that SibSp = 1 becomes more relevant.

#### Parch column and Survival

In [ ]:
all_unique_Parch = df['Parch'].unique()
for i in range(0, len(all_unique_Parch)):
    Parch_valv = df[df['Parch'] == all_unique_Parch[i]].reset_index(drop = True)
    probability_of_Parch_valv_class = len(Parch_valv[Parch_valv['Survived'] == 1])/len(Parch_valv)
    print(f"Parch -> {all_unique_Parch[i]}, Probability of surviving -> {probability_of_Parch_valv_class*100}")
    
#Confirming with Chi-Squared-test (Test for Independence)
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df["Parch"], df["Survived"])
_, p, _, expected = chi2_contingency(contingency_table)
print("p-value =", p)

#Performing Standard Residual to confirm that our observation is correct
std_residual = (contingency_table - expected)/np.sqrt(expected)
print(std_residual)

Inference
1. Using chi-squared test for independence of variables, we got p-value = 9.70e-05 which tell high dependence of Parch with the Survived column.
2. Using the standard residual column, there is significant relationship when Parch = 1, other than that, we are not able to see very high levels of significance with other values of SibSp.
    - In future while doing feature engineering, we might change the feature in some way so that Parch = 1 becomes more relevant.

#### Family size column and Survival

In [ ]:
all_unique_Parch = df['Total Family Size'].unique()
for i in range(0, len(all_unique_Parch)):
    Parch_valv = df[df['Total Family Size'] == all_unique_Parch[i]].reset_index(drop = True)
    probability_of_Parch_valv_class = len(Parch_valv[Parch_valv['Survived'] == 1])/len(Parch_valv)
    print(f"Total Family Size -> {all_unique_Parch[i]}, Probability of surviving -> {probability_of_Parch_valv_class*100}")
    
#Confirming with Chi-Squared-test (Test for Independence)
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df["Total Family Size"], df["Survived"])
_, p, _, expected = chi2_contingency(contingency_table)
print("p-value =", p)

#Performing Standard Residual to confirm that our observation is correct
std_residual = (contingency_table - expected)/np.sqrt(expected)
print(std_residual)

Inference
1. Using chi-squared test for independence of variables, we got p-value = 8.15e-10 which tell high dependence of Total Family Size with the Survived column.
2. Using the standard residual column, there is significant relationship.
3. **Total Family Size has shown much more relation with the Survived column as comparative to Parch and SibSp. Will keep this feature rather than 2 seperate Parch and SibSp** 

#### Embarked (all columns) and Survival

In [ ]:
def embarked_all_columns_p_value(column_name):
    #Confirming with Chi-Squared-test (Test for Independence)
    from scipy.stats import chi2_contingency
    contingency_table = pd.crosstab(df[column_name], df["Survived"])
    _, p, _, expected = chi2_contingency(contingency_table)
    print(f"p-value {column_name} =", p)
    #Performing Standard Residual to confirm that our observation is correct
    std_residual = (contingency_table - expected)/np.sqrt(expected)
    print(std_residual)
    print('\n')
        
embarked_all_columns_p_value("Embarked")
embarked_all_columns_p_value("Embarked_Mode_Imputing")
embarked_all_columns_p_value("Embarked_Missing_Imputing")

Inference
1. Using chi-squared test for independence of variables, all 3 varaints of Embarked has shown high dependence with the Survived column.
2. Using the standard residual column, all 3 variants shown significant relationship when Embarked = C.
3. **Best column can be chosen during Model Evaluation process**

#### Age (All columns) and Survived Column

In [ ]:
def age_all_columns_evaulation(df, column_name):

    #Getting 2 samples
    sample1_survived = np.array(df[df['Survived'] == 1][column_name])
    sample2_not_survived = np.array(df[df['Survived'] == 0][column_name])

    #Confirming variance of 2 samples to be equivalent
    print(f"Variance between 2 samples : {np.var(sample1_survived, ddof=1)/np.var(sample2_not_survived, ddof=1)}")

    #Test of independence using the t-test
    from scipy.stats import ttest_ind
    _, p_value = ttest_ind(sample1_survived, sample2_not_survived, equal_var=False)
    print("p-value t-test: ", p_value)

    #Confirming with ANNOVA also
    from scipy.stats import f_oneway
    _, p_value = f_oneway(sample1_survived, sample2_not_survived)
    print("p-value ANNOVA: ", p_value)
    
    print('\n')
    
#Removing the nan values
df_nan_removed = df.dropna(subset = 'Age').reset_index(drop = True)
#Imputing with Mean value
print('NaN values removing : ')
age_all_columns_evaulation(df_nan_removed, "Age")
print('Mean Imputation : ')
age_all_columns_evaulation(df, "Age_Mean_Imputed")
print('Median Imputation : ')
age_all_columns_evaulation(df, "Age_Median_Imputed")

Inference
1. All columns of Age are showing significant relationship with "Survived" column.
2. Significance of relationship : NaN value Removing > Mean Value imputation > Median Value imputation.
3. **Best method can be chosen during model evaluation process**

#### Fare and Survived Column

In [ ]:
#Getting 2 samples
sample1_survived = np.array(df[df['Survived'] == 1]['Fare'])
sample2_not_survived = np.array(df[df['Survived'] == 0]['Fare'])

#Confirming variance of 2 samples to be equivalent
print(f"Variance between 2 samples : {np.var(sample1_survived, ddof=1)/np.var(sample2_not_survived, ddof=1)}")

#Test of independence using the t-test
from scipy.stats import ttest_ind
_, p_value = ttest_ind(sample1_survived, sample2_not_survived, equal_var=False)
print("p-value t-test:", p_value)

#Confirming with ANNOVA also
from scipy.stats import f_oneway
_, p_value = f_oneway(sample1_survived, sample2_not_survived)
print("p-value ANNOVA:", p_value)

Inference
1. Using Welch t-test and One-way ANNOVA for independence of variables, we can say that there is significant relationship between Fare and Survived column.

#### Feature engineering after Bivariate Analysis
1. We are dropping the `SibSp` and `Parch` column from the dataframe, due to better relationship of `Total Family Size` column which was formed by combining both columns.
2. Convert the `Total Family Size` to Categories and again checking the relationship with the `Survived` column using Chi-Squared test.

In [ ]:
#Dropping SibSp and Parch columns
df = df.drop(["SibSp", "Parch"], axis = 1)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

#Converting Total Family Size to categories
df['Total Family category'] = None
for i in range(0, len(df)):
    if df['Total Family Size'][i] <= 1:
        df['Total Family category'][i] = 'Alone'
    elif df['Total Family Size'][i] <= 4:
        df['Total Family category'][i] = 'Small Family'
    elif df['Total Family Size'][i] > 4:
        df['Total Family category'][i] = 'Large Family'
        
#Checking relationship using Chi Squared test
from scipy.stats import chi2_contingency
contingency_table = pd.crosstab(df['Total Family category'], df["Survived"])
_, p, _, expected = chi2_contingency(contingency_table)
print(f"p-value =", p)

#Dropping the Total Family Size column
df = df.drop(["Total Family Size"], axis = 1)

Inference : There is a significant relationship between new `Total Family category` column and `Survived` column

# Model Building Phase - Logistic Regression
In this phase, we will combine Multivariate Analysis, feature engineering and Logistic Regression model to predict and infere the `Survived` from the independent variables. It will be an iterative process where we will make a Stable (By carefully meeting all assumptions) and Accurate (By doing intensive feature engineering and Multivariate Analysis) Logistic Regression Model.

## Creating the "Best dataset to Solve Age and Embarked missing value problem"
We need to handle Missing values in `Age` and `Embarked` columns. To do so, we will use Extrinsic approch. Create different models taking One-Constant-at-a-time and evaluate results on the basis of accuracy.

### Age column Imputation solution
We chose to take `Embarked` constant while finding the best `Age` imputation. Create seperate datasets for different Age imputations.

In [ ]:
#Creating a copy_df keeping One column for Embarked 
copy_df = df.copy()
copy_df = copy_df.drop(["Embarked_Missing_Imputing", "Embarked"], axis = 1)

In [ ]:
#One-Hot encoding and Standardization for Categorical and Numerical columns respectively
copy_df_encoded = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Total Family category", "Embarked_Mode_Imputing"], drop_first=True)

In [ ]:
#DataFrame with NaN values removed
copy_df_age_NaN_removing = copy_df_encoded.drop(["Age_Mean_Imputed", "Age_Median_Imputed"], axis = 1).dropna(subset = 'Age').reset_index(drop = True)

#Dataframe with mean value imputation
copy_df_age_Mean_imputation = copy_df_encoded.drop(["Age", "Age_Median_Imputed"], axis = 1)

#Dataframe with median value imputation
copy_df_age_Median_imputation = copy_df_encoded.drop(["Age_Mean_Imputed", "Age"], axis = 1)

#Dataframe with K-Imputer
from sklearn.impute import KNNImputer
copy_df_encoded_kNN_imputer = copy_df_encoded.drop(['Age_Mean_Imputed', 'Age_Median_Imputed'], axis = 1)
knn_imputer_object = KNNImputer(n_neighbors=5)
copy_df_encoded_kNN_imputer_imputed = knn_imputer_object.fit_transform(copy_df_encoded_kNN_imputer)
copy_df_encoded_kNN_imputer_imputed = pd.DataFrame(copy_df_encoded_kNN_imputer_imputed, columns=copy_df_encoded_kNN_imputer.columns)
copy_df_age_KNN_imputation = copy_df_encoded_kNN_imputer_imputed

In [ ]:
def model_building(df):
    #Training and testing split
    from sklearn.model_selection import train_test_split
    X = df.drop("Survived", axis=1)
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    #Standardizing data
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Model building
    from sklearn.linear_model import LogisticRegression
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)

    #Prediction
    y_pred = log_reg.predict(X_test_scaled)
    y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

    #Model evaluation
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score
    accuracy = accuracy_score(y_test, y_pred)
    confusion_matrix = confusion_matrix(y_test, y_pred)
    precision = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[1][0])
    recall = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[0][1])
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_prob)
    
    return accuracy, f1, auc_roc

#Checking the Accuray, AUC_ROC and F1 Score for all 3 datasets
print(model_building(copy_df_age_NaN_removing))
print(model_building(copy_df_age_Mean_imputation))
print(model_building(copy_df_age_Median_imputation))
print(model_building(copy_df_age_KNN_imputation))

Inference : Best Accuracy, AUC_ROC and F1 Score is attained by removing the NaN values from the dataset

#### Feature Engineering
Since, removing NaN values from the "Age" column provides highest accuracy, we will drop the 19% of the NaN columns from the dataset.

In [ ]:
df = df.drop(["Age_Mean_Imputed", "Age_Median_Imputed"], axis = 1)
df = df.dropna(subset = "Age").reset_index(drop = True)

### Embarked column Imputation solution
We took the featured engineered `Age` column and now we will find best imputation method for `Embarked`.

In [ ]:
#Creating a copy_df
copy_df = df.copy()

In [ ]:
#DataFrame with NaN values removed
copy_df_NaN_removing = copy_df.drop(["Embarked_Mode_Imputing", "Embarked_Missing_Imputing"], axis = 1).dropna(subset = "Embarked").reset_index(drop = True)
copy_df_NaN_removing = pd.get_dummies(copy_df_NaN_removing, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

#Dataframe with mean value imputation
copy_df_Mode_imputation = copy_df.drop(["Embarked", "Embarked_Missing_Imputing"], axis = 1)
copy_df_Mode_imputation = pd.get_dummies(copy_df_Mode_imputation, columns=["Pclass", "Sex", "Embarked_Mode_Imputing", "Total Family category"], drop_first=True)

#Dataframe with K-Imputer
from sklearn.impute import KNNImputer
copy_df_kNN_imputer = copy_df.drop(['Embarked_Mode_Imputing', 'Embarked_Missing_Imputing'], axis = 1)
copy_df_kNN_imputer = pd.get_dummies(copy_df_kNN_imputer, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)
knn_imputer_object = KNNImputer(n_neighbors=5)
copy_df_kNN_imputer_imputed = knn_imputer_object.fit_transform(copy_df_kNN_imputer)
copy_df_kNN_imputer_imputed = pd.DataFrame(copy_df_kNN_imputer_imputed, columns=copy_df_kNN_imputer.columns)
copy_df_KNN_imputation = copy_df_kNN_imputer_imputed

In [ ]:
#Checking the Accuray, AUC_ROC and F1 Score for all 3 datasets
print(model_building(copy_df_NaN_removing))
print(model_building(copy_df_Mode_imputation))
print(model_building(copy_df_KNN_imputation))

Inference : Best Accuracy, AUC_ROC and F1 Score is attained by removing the NaN values from the dataset.

#### Feature Engineering
Since, removing NaN values from the "Embarked" column provides highest accuracy, we will drop NaN rows from the dataset.

In [ ]:
df = df.drop(["Embarked_Mode_Imputing", "Embarked_Missing_Imputing"], axis = 1)
df = df.dropna(subset = "Embarked").reset_index(drop = True)

## Analysing the assumptions of Logistic Regression
Since now we have a cleaned dataset, we now need to check assumptions of Linear Regression on that dataset.

Assumptions of Logistic Regression Model are:
1. The dependent column (`Survived`) should be categorical with 2 categories.
2. There should be "No Multicollinarity" between the independent variables.
3. **There should be linear relationship between log odds and all independent variables. This assumption is specifically for Numerical Column `Age` and `Fare`. It is also applicable on Label Encoded independent variables ehich in our case out `Pclass`. In case of One-Hot encoded variables the assumption is always met.**
4. Independence of Errors.

The First 2 assumptions needs to be checked before model building phase and next 2 can be checked after model building phase.

### Assumption 1
Since the output column in `Survived` which is clearly categorical with 2 categories - First assumption is met for Logistic Regression.

### Assumption 2
According to the "No Multicollinearity" assumption, there should be No linear relationship between independent columns. To check the multicollinarity in the independent variables, we will perform 2 tests:-
1. Visual inspection using Pearson correlation and Heat Map. Peason tells the strength of linear relationship between the variables.
2. The confirmation will be taken on the basis of quantitative measurement using Variance Inflation Factor (VIF) which tells the strength of linear relationship of One independent variable with all others.

#### Visual Inspection

In [ ]:
copy_df = df.copy()
copy_df = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

import seaborn as sns 
import matplotlib.pyplot as plt 
corr = copy_df.corr() 
sns.heatmap(corr, annot=True, cmap="coolwarm") 
plt.show()

#### VIF method

In [ ]:
copy_df = df.copy()
copy_df = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
X = copy_df.drop(columns=["Survived"])
X_const = add_constant(X)
vif_data = pd.DataFrame()
vif_data["Feature"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) 
                   for i in range(X_const.shape[1])]

print(vif_data) #Since VIF values of all the columns are < 5, we can assume "No Multicollinarity"

Inference : 
- In visual Inspection all abs(values) are below  0.8, which confirms that "No Multicollinearity between independent variables".
- We confirmed by using VIF, which also checks for linear relationship between independent columns using regression analysis. Since all VIF's are less than 5, we confirm "No Multicollinearity between independent variables".

### Assumption 3 
To check this, we will first create a logistic regression model and then check the linearity of log odds with the numeric columns. We will do this using:
1. Visual Inspection.
2. Using Pearson correlation for log odds and independent numeric variable.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

copy_df = df.copy()
copy_df = pd.get_dummies(copy_df, columns=["Pclass", "Sex", "Embarked", "Total Family category"], drop_first=True)

def model_building(df):
    #Training and testing split
    from sklearn.model_selection import train_test_split
    X = df.drop("Survived", axis=1)
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    #Standardizing data
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Model building
    from sklearn.linear_model import LogisticRegression
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)

    #Prediction
    y_prob = log_reg.predict_proba(X_train_scaled)[:, 1]
    
    return y_prob, X_train

output_prob, X_train = model_building(copy_df)

In [ ]:
# To avoid division by zero or log(0), we can add a small epsilon
epsilon = 1e-10
log_odds_value = np.log((output_prob + epsilon) / (1 - (output_prob + epsilon)))

def plot_graph(x_column, logit_values, name_column):
    
    # Plotting the relationship
    plt.figure(figsize=(6, 4))
    sns.scatterplot(x=x_column, y=logit_values, alpha=0.5)
    sns.regplot(x=logit_values, y=logit_values, scatter=False, lowess=True, line_kws={'color': 'red'})

    plt.title(f"Linearity Check: {name_column} vs. Logit")
    plt.xlabel(name_column)
    plt.ylabel('Log-Odds (log(p/(1-p)))')
    plt.grid(True)
    plt.show()
    
plot_graph(X_train["Age"], log_odds_value, "Age")
plot_graph(X_train["Fare"], log_odds_value, "Fare")

In [ ]:
from scipy.stats import pearsonr
correlation_age, p_value_age = pearsonr(X_train["Age"], output_prob)
correlation_fare, p_value_fare = pearsonr(X_train['Fare'], output_prob)
print(p_value_age, p_value_fare)

Inference : 
With the help of visual inspection as well as pearson correlation, we are able to say the logit is significantly a linear relation of independent numeric columns.

## Creating the final model

In [ ]:
def model_building(df):
    #Training and testing split
    from sklearn.model_selection import train_test_split
    X = df.drop("Survived", axis=1)
    y = df["Survived"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    #Standardizing data
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    #Model building
    from sklearn.linear_model import LogisticRegression
    log_reg = LogisticRegression(max_iter=1000)
    log_reg.fit(X_train_scaled, y_train)

    #Prediction
    y_pred = log_reg.predict(X_test_scaled)
    y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

    #Model evaluation
    from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, roc_auc_score
    accuracy = accuracy_score(y_test, y_pred)
    confusion_matrix = confusion_matrix(y_test, y_pred)
    precision = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[1][0])
    recall = confusion_matrix[0][0]/(confusion_matrix[0][0]+confusion_matrix[0][1])
    f1 = f1_score(y_test, y_pred)
    auc_roc = roc_auc_score(y_test, y_prob)
    
    return accuracy, confusion_matrix, precision, recall, f1, auc_roc, log_reg, X.columns

df = pd.get_dummies(df, columns=["Pclass", "Sex", "Total Family category", "Embarked"], drop_first=True)
accuracy, confusion_matrix, precision, recall, f1, auc_roc, log_reg, input_features = model_building(df)

Inference:
- The model has fulfilled all the assumptions of the Logistic Regression model. 
- The model evaluation : 
    - Accuracy : 84.45%,
    -  confusion matrix : ([[73,  5],
      [16, 41]]),
    -  precision : 82.02%,
    -  recall : 93.58%,
    -  f1 score : 79.615,
    -  auc_roc score : 90.55%

# Model Inference - Logistic Regression

In [ ]:
# Create a DataFrame for interpretation
interpretation_df = pd.DataFrame({
    'Feature': input_features,
    'Coefficient': log_reg.coef_[0]
})

# Calculate Odds Ratio
interpretation_df['Odds Ratio'] = np.exp(interpretation_df['Coefficient'])

# Sort by Odds Ratio to see the most influential factors
interpretation_df = interpretation_df.sort_values(by='Odds Ratio', ascending=False)

## Final Model Interpretation: Key Factors for Survival on the Titanic : 
The logistic regression model has successfully identified several key factors that influenced a passenger's chances of survival. By converting the model's coefficients into Odds Ratios, we can clearly see the magnitude and direction of each factor's impact.

1. Sex (Male vs. Female):
- Odds Ratio for Sex_male: 0.318
- Inference: This is the most powerful predictor in the model. The odds of survival for a male passenger were only 0.318 times the odds for a female passenger, assuming all other factors (like class and age) were the same. To put it another way, a man's odds of surviving were about 68% lower than a woman's. This confirms the historical "women and children first" protocol.

2. Passenger Class (Pclass):
- Odds Ratio for Pclass_3: 0.359 (Compared to 1st Class)
- Odds Ratio for Pclass_2: 0.675 (Compared to 1st Class)
- Inference: Passenger class was the second most critical factor. The odds of survival for a passenger in 3rd class were about 64% lower than for a passenger in 1st class. The odds for a 2nd class passenger were 32.5% lower than for a 1st class passenger.
- Conclusion: There was a clear survival hierarchy based on wealth and status. First-class passengers had the highest chance of survival, followed by second, and then third class.

3. Age:
- Odds Ratio for Age: 0.535
- Inference: Age played a very significant role. For each additional year of a passenger's age, their odds of survival decreased by about 46.5% (1 - 0.535), holding all other factors constant. This strongly suggests that younger passengers had a much higher likelihood of surviving.

4. Family Size (Large vs. Alone):
- Odds Ratio for Total Family category_Large Family: 0.617
- Inference: Passengers traveling in a large family (more than 4 members) had 38% lower odds of survival compared to those traveling alone. This might be because it was more difficult for large families to stay together and evacuate quickly.

5. Fare:
- Odds Ratio for Fare: 1.103
- Inference: While less impactful than class, the fare paid still mattered. For every one-unit increase in fare (after standardization), a passenger's odds of survival increased by about 10%. This reinforces the idea that wealth was linked to survival, likely because higher fares corresponded to better cabin locations and access to lifeboats.

6. Point of Embarkation:
- Odds Ratio for Embarked_Q: 0.881 (Compared to Cherbourg)
- Odds Ratio for Embarked_S: 0.909 (Compared to Cherbourg)
- Inference: Passengers who boarded in Queenstown (Q) and Southampton (S) had slightly lower odds of survival (about 12% and 9% lower, respectively) compared to those who boarded in Cherbourg (C). This is a less direct factor and is likely correlated with other variables, such as the fact that a higher proportion of 1st class passengers boarded at Cherbourg.

7. Family Size (Small vs. Alone):
- Odds Ratio for Total Family category_Small Family: 0.995
- Inference: An odds ratio so close to 1 indicates that traveling in a small family (2-4 members) had almost no impact on survival odds compared to traveling alone.

### Summary of a Survivor's Profile
Based on the model, the passenger with the highest probability of surviving the Titanic disaster was a wealthy, young, female passenger from 1st class who boarded at Cherbourg and was not part of a large family. Conversely, the passenger with the lowest chance was an older, male passenger from 3rd class who was part of a large family.